# Laboratorio 2 : Programando una Mini Blockchain


Esta práctica avanzada te guiará paso a paso en la construcción de los componentes fundamentales de una blockchain educativa.
Aplicaremos programación Python para entender los conceptos de hash, transacciones firmadas, bloques, prueba de trabajo (Proof of Work),
verificación de cadenas y validación descentralizada.




In [24]:
import hashlib
import time
import json
import rsa

In [25]:
def generar_claves():
    """Genera un par de claves RSA para las billeteras."""
    (pubkey, privkey) = rsa.newkeys(512)
    return pubkey, privkey

In [26]:
class Transaccion:
    def __init__(self, emisor, receptor, cantidad):
        self.emisor = emisor
        self.receptor = receptor
        self.cantidad = cantidad
        self.firma = None

    def calcular_hash(self):
        # Convertimos las claves públicas a string si no son None (caso de recompensa)
        emisor_str = str(self.emisor.save_pkcs1()) if self.emisor else "Red"
        receptor_str = str(self.receptor.save_pkcs1())
        tx_string = f"{emisor_str}{receptor_str}{self.cantidad}"
        return hashlib.sha256(tx_string.encode()).hexdigest()

    def firmar(self, clave_privada):
        if self.emisor is None:
            return  # Las transacciones de recompensa de minado no se firman

        hash_tx = self.calcular_hash().encode()
        self.firma = rsa.sign(hash_tx, clave_privada, 'SHA-256')

    def es_valida(self):
        if self.emisor is None:
            return True # Transacción de minado (Génesis o Recompensa)

        if not self.firma:
            return False

        try:
            hash_tx = self.calcular_hash().encode()
            rsa.verify(hash_tx, self.firma, self.emisor)
            return True
        except rsa.VerificationError:
            return False

In [27]:
class Bloque:
    def __init__(self, transacciones, hash_previo=""):
        self.timestamp = time.time()
        self.transacciones = transacciones
        self.hash_previo = hash_previo
        self.nonce = 0 # Paso 4: El nonce se usará para variar el hash
        self.hash = self.calcular_hash()

    def calcular_hash(self):
        tx_hashes = [tx.calcular_hash() for tx in self.transacciones]
        bloque_string = f"{self.timestamp}{tx_hashes}{self.hash_previo}{self.nonce}"
        return hashlib.sha256(bloque_string.encode()).hexdigest()

    def minar_bloque(self, dificultad):
        objetivo = "0" * dificultad
        while self.hash[:dificultad] != objetivo:
            self.nonce += 1
            self.hash = self.calcular_hash()
        print(f"Bloque minado! Hash: {self.hash}")

In [28]:
class Blockchain:
    def __init__(self, dificultad=2):
        self.cadena = [self.crear_bloque_genesis()] # Paso 7: Construir cadena
        self.dificultad = dificultad
        self.transacciones_pendientes = []
        self.recompensa_minado = 50  # Paso 6: Recompensa por minar

    def crear_bloque_genesis(self):
        # El bloque génesis se crea manualmente (Responde a tu pregunta inicial)
        tx_genesis = Transaccion(None, rsa.newkeys(512)[0], 0) # Transacción nula
        bloque = Bloque([tx_genesis], "0")
        # No minamos el bloque génesis, su hash no tiene ceros a la izquierda
        return bloque

    def obtener_ultimo_bloque(self):
        return self.cadena[-1]

    def agregar_transaccion(self, transaccion):
        if not transaccion.es_valida():
            raise Exception("Transacción inválida. No se puede agregar.")
        self.transacciones_pendientes.append(transaccion)

    def minar_transacciones_pendientes(self, direccion_minero):
        # Creamos la transacción de recompensa
        tx_recompensa = Transaccion(None, direccion_minero, self.recompensa_minado)
        self.transacciones_pendientes.append(tx_recompensa)

        bloque = Bloque(self.transacciones_pendientes, self.obtener_ultimo_bloque().hash)
        bloque.minar_bloque(self.dificultad)

        print("Bloque agregado exitosamente a la cadena.")
        self.cadena.append(bloque)
        self.transacciones_pendientes = [] # Limpiamos la lista

    def es_cadena_valida(self, cadena_a_verificar=None):
        cadena = cadena_a_verificar if cadena_a_verificar else self.cadena

        for i in range(1, len(cadena)):
            bloque_actual = cadena[i]
            bloque_previo = cadena[i-1]

            # Verificar si el hash del bloque es correcto
            if bloque_actual.hash != bloque_actual.calcular_hash():
                return False

            # Verificar si el bloque apunta correctamente al anterior
            if bloque_actual.hash_previo != bloque_previo.hash:
                return False

            # Verificar las firmas de las transacciones
            for tx in bloque_actual.transacciones:
                if not tx.es_valida():
                    return False
        return True

    def obtener_saldo(self, direccion):
        saldo = 0
        for bloque in self.cadena:
            for tx in bloque.transacciones:
                if tx.emisor == direccion:
                    saldo -= tx.cantidad
                if tx.receptor == direccion:
                    saldo += tx.cantidad
        return saldo

    def resolver_conflictos(self, otra_cadena):
        if len(otra_cadena) > len(self.cadena) and self.es_cadena_valida(otra_cadena):
            print("Nuestra cadena ha sido reemplazada por una más larga.")
            self.cadena = otra_cadena
        else:
            print("Nuestra cadena es la autoridad. No se reemplaza.")

    def visualizar_cadena(self):
        print("\n--- ESTADO DE LA BLOCKCHAIN ---")
        for i, bloque in enumerate(self.cadena):
            print(f"Bloque {i}:")
            print(f"  Timestamp: {bloque.timestamp}")
            print(f"  Hash: {bloque.hash}")
            print(f"  Hash Previo: {bloque.hash_previo}")
            print(f"  Nonce: {bloque.nonce}")
            print(f"  Transacciones: {len(bloque.transacciones)}")
            print("-" * 30)

In [29]:
# Inicializar Blockchain -> Al iniciar la Blockchain trae implicito crear el bloque génesis

mi_moneda = Blockchain(dificultad=4)

## 1. Generar claves públicas y privadas con RSA

In [30]:
pub_daniel, priv_daniel = generar_claves()
pub_chelo, priv_chelo = generar_claves()
pub_minero, priv_minero = generar_claves()

In [31]:
# Simulamos que mine un bloque para obtener la recompensa y poder realizar transacciones
mi_moneda.minar_transacciones_pendientes(pub_daniel)

Bloque minado! Hash: 0000ee616af8a274a8aa361b0a0cd8a87288de7be0b765516f9643e50065e486
Bloque agregado exitosamente a la cadena.


## 2. Crear y firmar una transacción digital

In [32]:
tx1 = Transaccion(pub_daniel, pub_chelo, 15)
tx1.firmar(priv_daniel)

## 3. Verificar la firma de una transacción

In [33]:
es_firma_valida = tx1.es_valida()
print(f"¿La firma de la transacción es válida?: {es_firma_valida}")

# Si es válida, la agregamos a las transacciones pendientes
if es_firma_valida:
  mi_moneda.agregar_transaccion(tx1)
  print("Transacción agregada a la lista de tareas pendientes (Mempool).")

¿La firma de la transacción es válida?: True
Transacción agregada a la lista de tareas pendientes (Mempool).


## 4 .Implementar un bloque con hash y nonce

## 5. Aplicar Proof of Work para encontrar un hash válido

## 6. Simular una recompensa por minado (bloque con recompensa)

## 7. Construir una cadena de bloques (blockchain)

In [34]:
# Al llamar a esta función, internamente se crea el Bloque (Paso 4),
# se busca el hash válido iterando el nonce (Paso 5), se añade la recompensa (Paso 6)
# y se une a la cadena principal (Paso 7).

mi_moneda.minar_transacciones_pendientes(pub_minero)

Bloque minado! Hash: 00008640d79167afdf0377261955b9ac38a679751b29eee70774945ad9fb8904
Bloque agregado exitosamente a la cadena.


## 8.  Validar la integridad de toda la cadena

In [35]:
es_integra = mi_moneda.es_cadena_valida()
print(f"¿La blockchain es válida e íntegra en este momento?: {es_integra}")

¿La blockchain es válida e íntegra en este momento?: True


## 9. Consultar el saldo de una dirección

In [36]:
print(f"Saldo de Daniel: {mi_moneda.obtener_saldo(pub_daniel)}")
print(f"Saldo de Chelo: {mi_moneda.obtener_saldo(pub_chelo)}")
print(f"Saldo del Minero: {mi_moneda.obtener_saldo(pub_minero)}")

Saldo de Daniel: 35
Saldo de Chelo: 15
Saldo del Minero: 50


## 10. Simular conflicto entre bloques y resolverlo con 'la cadena más larga'

In [37]:
# Creamos una blockchain nueva (que solo tiene el bloque génesis) para simular que
# un nodo intenta imponernos una cadena más corta.
cadena_alternativa = Blockchain().cadena

mi_moneda.resolver_conflictos(cadena_alternativa)

Nuestra cadena es la autoridad. No se reemplaza.


##11. Visualización de la cadena de bloques

In [38]:
mi_moneda.visualizar_cadena()


--- ESTADO DE LA BLOCKCHAIN ---
Bloque 0:
  Timestamp: 1772796491.0675478
  Hash: 6c808b7cbce3bf60c7c331770aa0f11d4ce4d253952993b8abee85d58bfebb07
  Hash Previo: 0
  Nonce: 0
  Transacciones: 1
------------------------------
Bloque 1:
  Timestamp: 1772796491.242907
  Hash: 0000ee616af8a274a8aa361b0a0cd8a87288de7be0b765516f9643e50065e486
  Hash Previo: 6c808b7cbce3bf60c7c331770aa0f11d4ce4d253952993b8abee85d58bfebb07
  Nonce: 34305
  Transacciones: 1
------------------------------
Bloque 2:
  Timestamp: 1772796493.0259597
  Hash: 00008640d79167afdf0377261955b9ac38a679751b29eee70774945ad9fb8904
  Hash Previo: 0000ee616af8a274a8aa361b0a0cd8a87288de7be0b765516f9643e50065e486
  Nonce: 6727
  Transacciones: 2
------------------------------


## 12. ¿Por qué el primer bloque no cumple con el prefijo 0000 en el hash?


El primer bloque de una blockchain se denomina Bloque Génesis y tiene una naturaleza especial: no se mina.

Para que la Prueba de Trabajo (Proof of Work) exija ceros a la izquierda, el minero debe iterar el nonce múltiples veces hasta dar con el hash objetivo. Sin embargo, el Bloque Génesis es creado de forma manual (está preprogramado o hardcoded) en el código fuente en el instante en que nace la red.

Como no compite en la red ni necesita consenso para existir (es la base indiscutible sobre la que se construye el resto), su hash es simplemente el resultado de encriptar sus datos en el primer intento, sin aplicarle la restricción de dificultad de la red.